# Hawaiian Newspaper OCR → Reference Audit → Translation

This notebook builds a pipeline for working with scanned **Hawaiian-language** newspapers:

1. **Load** a PDF of page images and render each page to a high-resolution image.
2. **OCR** the text with **Tesseract** using the Hawaiian language model (`haw`).
3. **Audit** a reference transcription you provide (e.g. from the Papakilo / Nūpepa database) against what is actually on the page — flagging where the reference disagrees with the page so you can correct the reference.
4. **Translate** the corrected Hawaiian text to English (free Google Translate by default; Claude optionally, if you have an API key).

### The page is the source of truth
The scanned page is the primary source. The reference text is a transcription that **may contain errors** — the goal of step 3 is to find those errors, not to trust the reference. Because OCR is itself imperfect, a flagged discrepancy is a *candidate* to check against the page image: some flags will be genuine reference errors, some will be OCR slips. You adjudicate.

Tesseract's Hawaiian model recognizes the diacritics — the ʻokina (ʻ) and the kahakō / macrons (ā ē ī ō ū) — which makes the page-vs-reference comparison meaningful.

### Before you start
- Install the Tesseract engine + Hawaiian model (Step 0b).
- Put your newspaper PDF in this folder (default name `newspaper.pdf`).
- Have the reference transcription ready to paste in Step 5.
- Translation uses Google Translate by default (no key needed). To use Claude instead (higher quality on archaic Hawaiian), set `export ANTHROPIC_API_KEY=...` and `TRANSLATION_BACKEND = "claude"`.

Run the cells top to bottom.

## Step 0 — Install Python dependencies

Run once. These are the Python packages; the Tesseract engine itself is installed separately in Step 0b.

In [ ]:
# Install the Python packages. Safe to re-run; pip skips what's present.
%pip install --quiet pymupdf pytesseract opencv-python-headless pillow numpy deep-translator anthropic
print("Python dependencies installed.")

## Step 0b — Install the Tesseract engine + Hawaiian model

Tesseract is a system program, not a pip package. Install it and the language data, which includes the Hawaiian model (`haw`):

```
# macOS
brew install tesseract tesseract-lang

# Debian / Ubuntu
sudo apt-get install tesseract-ocr tesseract-ocr-haw
```

Run that in a terminal (or in a notebook cell with a leading `!`), then run the check below to confirm `haw` is available.

In [ ]:
import shutil, subprocess

exe = shutil.which("tesseract")
if not exe:
    print("Tesseract not found on PATH. Install it (see above), then restart the kernel.")
    print("If it's installed in a custom location, set TESSERACT_CMD in the config cell.")
else:
    langs = subprocess.run([exe, "--list-langs"], capture_output=True, text=True).stdout
    print("tesseract:", exe)
    print("Hawaiian model ('haw') present:", "haw" in langs.split())
    print(langs)

## Step 1 — Configuration

Edit the values below to point at your files and pick your options.

In [ ]:
import os
import re
import difflib
from pathlib import Path

import numpy as np
from PIL import Image

# ---------------------------------------------------------------------------
# Configuration — edit these
# ---------------------------------------------------------------------------
PDF_PATH       = "newspaper.pdf"      # the scanned newspaper PDF (place it in this folder)
REFERENCE_PATH = "reference.txt"      # optional file fallback for the reference text
OUTPUT_DIR     = "output"             # where extracted text / reports / translations are written

RENDER_DPI = 300                      # higher = sharper images = better OCR (but slower)
PREPROCESS = True                     # apply grayscale / threshold / deskew before OCR

OCR_LANG      = "haw"                 # Tesseract language; "haw" = Hawaiian. Use "haw+eng" for mixed pages.
TESSERACT_CMD = ""                    # optional: full path to the tesseract binary if it's not on PATH

# Translation
TRANSLATION_BACKEND = "google"        # "google" (free, no key), "claude" (needs ANTHROPIC_API_KEY), or "auto"
CLAUDE_MODEL = "claude-opus-4-8"      # translation model (only used if backend is claude)
SOURCE_LANG = "haw"                   # Hawaiian (used by the Google backend)

# Set in Step 5 (paste cell); kept here so the audit cell never hits a NameError.
REFERENCE_TEXT = ""

Path(OUTPUT_DIR).mkdir(exist_ok=True)
print("Config loaded. Output dir:", os.path.abspath(OUTPUT_DIR))

## Step 2 — Load the PDF and render pages to images

We render each page at `RENDER_DPI` with PyMuPDF. Rendering (rather than extracting embedded images) works for both scanned and vector PDFs.

In [ ]:
import fitz  # PyMuPDF

def pdf_to_images(pdf_path, dpi=300):
    # Render each page of a PDF to a PIL RGB image.
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(
            f"Could not find '{pdf_path}'. Put your PDF in this folder or update PDF_PATH."
        )
    doc = fitz.open(pdf_path)
    zoom = dpi / 72.0  # 72 is the PDF base DPI
    mat = fitz.Matrix(zoom, zoom)
    images = []
    for page in doc:
        pix = page.get_pixmap(matrix=mat, alpha=False)
        images.append(Image.frombytes("RGB", (pix.width, pix.height), pix.samples))
    doc.close()
    return images

pages = pdf_to_images(PDF_PATH, dpi=RENDER_DPI)
print(f"Rendered {len(pages)} page(s) at {RENDER_DPI} DPI.")

# Preview the first page (downscaled for display)
preview = pages[0].copy()
preview.thumbnail((700, 900))
preview

## Step 3 — (Optional) Preprocess images for better OCR

Old newsprint is yellowed, grainy, and often slightly skewed. Grayscale + denoise + adaptive threshold + deskew gives Tesseract clean black-on-white text, which usually helps. Set `PREPROCESS = False` in the config if it hurts on your scans.

In [ ]:
import cv2

def preprocess(pil_img):
    # Grayscale + denoise + adaptive threshold + deskew. Returns a PIL image.
    img = np.array(pil_img.convert("L"))            # grayscale
    img = cv2.fastNlMeansDenoising(img, h=10)       # reduce paper grain
    # Adaptive threshold handles uneven lighting / yellowed paper
    bw = cv2.adaptiveThreshold(
        img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 15
    )
    # Deskew based on the dominant text angle
    coords = np.column_stack(np.where(bw < 255)).astype(np.float32)
    if len(coords) > 0:
        angle = cv2.minAreaRect(coords)[-1]
        if angle < -45:
            angle = 90 + angle
        if abs(angle) > 0.5:
            h, w = bw.shape
            M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
            bw = cv2.warpAffine(bw, M, (w, h),
                                flags=cv2.INTER_CUBIC,
                                borderMode=cv2.BORDER_REPLICATE)
    return Image.fromarray(bw)

if PREPROCESS:
    sample = preprocess(pages[0])
    prev = sample.copy(); prev.thumbnail((700, 900))
    display(prev)
else:
    print("Preprocessing disabled (PREPROCESS = False).")

## Step 4 — OCR with Tesseract

Tesseract's LSTM engine runs with the Hawaiian model (`haw`), so it recognizes ʻokina and kahakō (ā ē ī ō ū) directly. This OCR is our reading of what is actually printed on the page — the evidence we audit the reference against in Step 5.

Tune `OCR_LANG` (e.g. `"haw+eng"` for mixed pages) and `TESS_CONFIG` (the `--psm` page-segmentation mode) below if needed. Multi-column layouts may be read out of order; that mostly affects ordering, not the word-level audit.

In [ ]:
import pytesseract

if TESSERACT_CMD:
    pytesseract.pytesseract.tesseract_cmd = TESSERACT_CMD

# --oem 1 = LSTM engine; --psm 3 = fully automatic page segmentation.
TESS_CONFIG = "--oem 1 --psm 3"

def ocr_image(pil_img):
    # Return recognized text for one page image.
    return pytesseract.image_to_string(pil_img, lang=OCR_LANG, config=TESS_CONFIG)

ocr_pages = []
for i, page in enumerate(pages, start=1):
    img = preprocess(page) if PREPROCESS else page
    text = ocr_image(img)
    ocr_pages.append(text)
    print(f"[page {i}/{len(pages)}] {len(text)} chars")

ocr_text = "\n\n".join(ocr_pages)

ocr_out = Path(OUTPUT_DIR) / "extracted_ocr.txt"
ocr_out.write_text(ocr_text, encoding="utf-8")
print(f"\nSaved raw OCR to {ocr_out}\n")
print("----- preview -----")
print(ocr_text[:1500])

## Step 5 — provide the reference text to audit

Paste the reference transcription (e.g. the text shown on the Papakilo article page) into `REFERENCE_TEXT` below, then run it. This is the text we will check for errors — re-run with new text for each article.

If you correct the reference after reviewing the discrepancies, paste the fixed text back here and re-run the audit.

In [ ]:
# === Paste the reference transcription to audit here, then run this cell. ===
# Replace the placeholder line between the ''' markers. Re-run for each new article.
REFERENCE_TEXT = '''
PASTE THE HAWAIIAN REFERENCE TEXT HERE
'''

print(len(REFERENCE_TEXT.strip()), "chars of reference text loaded.")

### Audit the reference against the page

Neither side is assumed correct. This step:

- **Normalizes** both texts (unifies ʻokina variants, collapses whitespace) so only real differences surface.
- **Aligns** the reference against the page OCR word-by-word and **flags every discrepancy** — `DIFFERS` (the two read differently), `PAGE HAS / REFERENCE LACKS` (the reference may be missing text), and `REFERENCE HAS / PAGE LACKS` (the reference may have added text the page doesn't show).
- Writes a numbered discrepancy report and an annotated copy of the reference, and shows a visual diff.

Expect some flags to be OCR noise — verify each against the page image before changing the reference.

In [ ]:
# Resolve the reference text: pasted REFERENCE_TEXT wins, then reference.txt fallback.
if REFERENCE_TEXT.strip():
    reference_text = REFERENCE_TEXT
elif os.path.exists(REFERENCE_PATH):
    reference_text = Path(REFERENCE_PATH).read_text(encoding="utf-8")
else:
    reference_text = ""
    print("No reference text. Paste it into REFERENCE_TEXT in the cell above to audit it.")

# ---------------------------------------------------------------------------
# Hawaiian-aware normalization (unify ʻokina variants, collapse whitespace).
# ---------------------------------------------------------------------------
OKINA = "ʻ"  # U+02BB
APOSTROPHE_VARIANTS = ["'", "’", "‘", "`", "´", "ʼ"]

def normalize(text):
    for ch in APOSTROPHE_VARIANTS:
        text = text.replace(ch, OKINA)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def audit_reference(reference, ocr):
    # Compare the reference (a=reference) against the page OCR (b=page). Neither is
    # assumed correct; every mismatch is returned for you to verify against the image.
    ref = normalize(reference).split()
    pg = normalize(ocr).split()
    sm = difflib.SequenceMatcher(a=ref, b=pg, autojunk=False)
    discrepancies, annotated = [], []
    CTX = 4
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "equal":
            annotated.extend(ref[i1:i2])
            continue
        ref_span = " ".join(ref[i1:i2])
        pg_span = " ".join(pg[j1:j2])
        before = " ".join(ref[max(0, i1 - CTX):i1])
        after = " ".join(ref[i2:i2 + CTX])
        discrepancies.append({
            "type": tag,                  # replace | delete (ref-only) | insert (page-only)
            "reference": ref_span,
            "page_ocr": pg_span,
            "context": f"...{before} [HERE] {after}...",
        })
        if tag == "replace":
            annotated.append(f"⟦REF:{ref_span} | PAGE:{pg_span}⟧")
        elif tag == "delete":
            annotated.append(f"⟦REF-ONLY:{ref_span}⟧")
        elif tag == "insert":
            annotated.append(f"⟦PAGE-ONLY:{pg_span}⟧")
    return discrepancies, " ".join(annotated), round(sm.ratio(), 4)

KIND = {
    "replace": "DIFFERS",
    "delete": "REFERENCE HAS / PAGE LACKS",
    "insert": "PAGE HAS / REFERENCE LACKS",
}

if reference_text.strip():
    discrepancies, annotated_ref, similarity = audit_reference(reference_text, ocr_text)
    print(f"Reference vs page-OCR similarity: {similarity}")
    print(f"{len(discrepancies)} discrepancy span(s) flagged for review.\n")

    lines = []
    for n, d in enumerate(discrepancies, 1):
        lines.append(
            f"[{n}] {KIND[d['type']]}\n"
            f"    reference: {d['reference'] or '(none)'}\n"
            f"    page OCR : {d['page_ocr'] or '(none)'}\n"
            f"    near     : {d['context']}\n"
        )
    report = "\n".join(lines) if lines else "No discrepancies found."
    (Path(OUTPUT_DIR) / "discrepancies.txt").write_text(report, encoding="utf-8")
    (Path(OUTPUT_DIR) / "reference_annotated.txt").write_text(annotated_ref, encoding="utf-8")
    print(report[:2000])
    print("\nSaved discrepancies.txt and reference_annotated.txt to", OUTPUT_DIR)
else:
    annotated_ref = ""
    discrepancies = []

In [ ]:
# Visual diff: reference (left) vs page OCR (right). Highlighted spans are the discrepancies.
from IPython.display import HTML, display

def audit_diff(reference, ocr, full=False):
    a = normalize(reference).split("\n")
    b = normalize(ocr).split("\n")
    differ = difflib.HtmlDiff(wrapcolumn=80)
    maker = differ.make_file if full else differ.make_table
    return maker(a, b, fromdesc="Reference (under review)", todesc="Page OCR (Tesseract)",
                 context=True, numlines=2)

if reference_text.strip():
    (Path(OUTPUT_DIR) / "diff.html").write_text(
        audit_diff(reference_text, ocr_text, full=True), encoding="utf-8")
    display(HTML(audit_diff(reference_text, ocr_text)))
    print("Full diff saved to", Path(OUTPUT_DIR) / "diff.html")
else:
    print("Provide reference text to see a diff.")

In [ ]:
# Choose the text to carry forward to translation.
# Default = your reference transcription. After you review the discrepancies and fix
# the reference (re-run the paste + audit cells), this updates automatically. Or set
# final_text to something else here (e.g. the OCR text, or a Claude-adjudicated version).
final_text = reference_text if reference_text.strip() else normalize(ocr_text)
print(f"final_text set ({len(final_text)} chars). Edit the reference and re-run to update it.")

### Step 5b — (Optional) Claude-assisted adjudication

If you have an `ANTHROPIC_API_KEY`, Claude can weigh the reference against the page OCR and propose a corrected reference, restoring ʻokina and kahakō. (Better still, use the Claude vision cell at the end to read the actual image.) Uncomment the bottom block to run.

In [ ]:
def claude_adjudicate(reference, ocr, model=CLAUDE_MODEL):
    import anthropic
    client = anthropic.Anthropic()
    system = (
        "You are auditing a Hawaiian (ʻŌlelo Hawaiʻi) newspaper transcription. "
        "(A) is a REFERENCE transcription that may contain errors. "
        "(B) is OCR read directly from the scanned page. Neither is fully reliable. "
        "Using both as evidence, produce the most accurate Hawaiian text, correcting errors "
        "in the reference where the page clearly disagrees. Restore ʻokina (ʻ) and kahakō "
        "(ā ē ī ō ū). Return ONLY the corrected Hawaiian text, no commentary."
    )
    user = f"=== A: REFERENCE ===\n{reference}\n\n=== B: PAGE OCR ===\n{ocr}"
    resp = client.messages.create(
        model=model, max_tokens=16000, system=system,
        messages=[{"role": "user", "content": user}],
    )
    return next((b.text for b in resp.content if b.type == "text"), "")

# Uncomment to run and carry the result forward to translation:
# if reference_text.strip() and os.environ.get("ANTHROPIC_API_KEY"):
#     final_text = claude_adjudicate(reference_text, ocr_text)
#     (Path(OUTPUT_DIR) / "corrected_claude.txt").write_text(final_text, encoding="utf-8")
#     print(final_text[:1500])
# else:
#     print("Set ANTHROPIC_API_KEY and provide reference text to use Claude adjudication.")

## Step 6 — Translate Hawaiian → English

Default backend is **Google Translate** (`deep-translator`) — free, no API key, needs internet. It supports Hawaiian but is rough on archaic newspaper text.

If you ever get an `ANTHROPIC_API_KEY`, set `TRANSLATION_BACKEND = "claude"` in the config for much better 19th-/early-20th-century Hawaiian translation. Long text is chunked automatically either way.

In [ ]:
# ---------------------------------------------------------------------------
# Translate Hawaiian -> English. Translates `final_text` (see the cell above).
# ---------------------------------------------------------------------------
def _chunk(text, size=4000):
    # Split text into <=size-char chunks on line boundaries.
    parts, buf = [], ""
    for line in text.split("\n"):
        if len(buf) + len(line) + 1 > size and buf:
            parts.append(buf); buf = ""
        buf += line + "\n"
    if buf.strip():
        parts.append(buf)
    return parts

def translate_google(text, source=SOURCE_LANG):
    # Free, no key, needs internet. Falls back to auto-detect if the code is rejected.
    from deep_translator import GoogleTranslator
    try:
        tr = GoogleTranslator(source=source, target="en")
        return "\n".join(tr.translate(c) for c in _chunk(text, size=4500))
    except Exception:
        tr = GoogleTranslator(source="auto", target="en")
        return "\n".join(tr.translate(c) for c in _chunk(text, size=4500))

def translate_claude(text, model=CLAUDE_MODEL):
    # Optional upgrade; requires ANTHROPIC_API_KEY.
    import anthropic
    client = anthropic.Anthropic()
    system = (
        "You are an expert translator of ʻŌlelo Hawaiʻi (Hawaiian) into English, "
        "specializing in 19th- and early-20th-century Hawaiian-language newspapers. "
        "Translate faithfully and naturally. Keep proper nouns and place names in Hawaiian. "
        "If a passage is garbled or uncertain, translate your best reading and mark it [unclear]. "
        "Return ONLY the English translation."
    )
    outs = []
    for chunk in _chunk(text, size=4000):
        resp = client.messages.create(
            model=model, max_tokens=16000, system=system,
            messages=[{"role": "user", "content": chunk}],
        )
        outs.append(next((b.text for b in resp.content if b.type == "text"), ""))
    return "\n".join(outs)

def translate(text):
    backend = TRANSLATION_BACKEND
    if backend == "auto":
        backend = "claude" if os.environ.get("ANTHROPIC_API_KEY") else "google"
    if backend == "claude" and not os.environ.get("ANTHROPIC_API_KEY"):
        print("No ANTHROPIC_API_KEY found - using Google Translate instead.")
        backend = "google"
    print(f"Translating with: {backend}")
    return translate_claude(text) if backend == "claude" else translate_google(text)

english = translate(final_text)
en_out = Path(OUTPUT_DIR) / "translation_en.txt"
en_out.write_text(english, encoding="utf-8")
print("Saved English translation to", en_out, "\n")
print("----- preview -----")
print(english[:1500])

In [ ]:
# Side-by-side view of the carried-forward Hawaiian and the English translation.
import html as _html
from IPython.display import HTML

def side_by_side(haw, eng):
    h = _html.escape(haw); e = _html.escape(eng)
    cell = "style='vertical-align:top;width:50%;padding:8px;border:1px solid #eee'"
    return HTML(
        "<table style='width:100%'><tr>"
        + f"<td {cell}><b>Hawaiian (carried forward)</b><pre style='white-space:pre-wrap'>{h}</pre></td>"
        + f"<td {cell}><b>English</b><pre style='white-space:pre-wrap'>{e}</pre></td>"
        + "</tr></table>"
    )

side_by_side(final_text, english)

### (Optional) Cross-check the page with Claude vision

Claude can read a page image directly — the best way to adjudicate a flagged discrepancy, since it looks at the actual page rather than the OCR. Requires `ANTHROPIC_API_KEY`.

In [ ]:
import base64, io

def claude_vision_ocr(pil_img, model=CLAUDE_MODEL):
    import anthropic
    client = anthropic.Anthropic()
    img = pil_img.copy(); img.thumbnail((1500, 1500))   # keep the request small
    buf = io.BytesIO(); img.save(buf, format="PNG")
    b64 = base64.standard_b64encode(buf.getvalue()).decode("utf-8")
    resp = client.messages.create(
        model=model, max_tokens=8000,
        messages=[{"role": "user", "content": [
            {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": b64}},
            {"type": "text", "text": (
                "Transcribe ALL Hawaiian text on this newspaper page exactly, "
                "including ʻokina (ʻ) and kahakō (ā ē ī ō ū). Return only the transcription."
            )},
        ]}],
    )
    return next((b.text for b in resp.content if b.type == "text"), "")

# Uncomment to run on the first page:
# if os.environ.get("ANTHROPIC_API_KEY"):
#     print(claude_vision_ocr(pages[0])[:1500])
# else:
#     print("Set ANTHROPIC_API_KEY to use the Claude vision cross-check.")

## Outputs & next steps

Everything is written to the `output/` folder:

| File | What it is |
|------|------------|
| `extracted_ocr.txt` | Raw Tesseract OCR text from the page images |
| `discrepancies.txt` | Every place the reference disagrees with the page OCR — your review list |
| `reference_annotated.txt` | The reference with each discrepancy marked inline (`⟦REF:… \| PAGE:…⟧`) |
| `diff.html` | Side-by-side visual diff: reference vs page OCR (open in a browser) |
| `translation_en.txt` | English translation of the text you carried forward |

**Workflow**
1. Run the audit, then review `discrepancies.txt` / the diff against the page image.
2. Fix the genuine errors in your reference text, paste it back into Step 5, and re-run the audit until only OCR-noise flags remain.
3. Translate the corrected text.

**Tuning tips**
- If OCR is weak (lots of false flags): raise `RENDER_DPI` (e.g. 400), toggle `PREPROCESS`, or change `TESS_CONFIG` (`--psm 4` or `--psm 6` can help on single-column clippings).
- For mixed Hawaiian/English pages set `OCR_LANG = "haw+eng"`.
- To adjudicate against the actual page automatically, use the optional Claude vision cell (needs an API key).